# Modelagem de Tópicos com BERTopic

Este notebook realiza a **modelagem de tópicos** em comentários do YouTube usando a biblioteca [BERTopic](https://maartengr.github.io/BERTopic/).

## Fluxo geral
1. Carrega os comentários de um arquivo CSV
2. Configura o modelo BERTopic para português
3. Treina o modelo e extrai os tópicos
4. (Opcional) Reduz comentários classificados como outlier (`-1`)
5. Salva o resultado em um novo CSV

## Requisitos
```
pip install bertopic sentence-transformers pandas
```

## 1. Imports

In [ ]:
import pandas as pd
from bertopic import BERTopic

## 2. Carregamento dos dados

In [ ]:
comentários_do_youtube = pd.read_csv ("ARQUIVO COM COMENTÁRIOS", encoding='utf-8-sig')  # <- altere para o caminho do seu arquivo

print(f"Total de linhas carregadas: {len(comentários_do_youtube)}")
comentários_do_youtube.head()

## 3. Pré-processamento

In [ ]:
# Remove linhas sem comentário
comentários_do_youtube = comentários_do_youtube.dropna(subset=["COLUNA COM COMENTÁRIOS"]).copy()

# Converte a coluna para lista de strings
documentos = comentários_do_youtube["COLUNA COM COMENTÁRIOS"].astype(str).tolist()

print(f"Comentários válidos para modelagem: {len(documentos)}")

## 4. Configuração do modelo

In [7]:
topic_model = BERTopic(
    embedding_model="paraphrase-multilingual-MiniLM-L12-v2", # usa o modelo `paraphrase-multilingual-MiniLM-L12-v2`, que suporta português e outros idiomas.
    language="multilingual",   # indica suporte a múltiplos idiomas
    calculate_probabilities=True,  # necessário para redução de outliers
    verbose=True               # exibe progresso durante o treinamento
)

## 5. Treinamento do modelo

In [ ]:
topics, probs = topic_model.fit_transform(documentos)

print(f"Número de tópicos encontrados: {topic_model.get_topic_info().shape[0] - 1}")  # -1 exclui outliers
print(f"Comentários classificados como outlier (-1): {topics.count(-1)}")

## 6. (Opcional) Redução de outliers

In [ ]:
# Extrai os embeddings dos documentos para calcular similaridade
embeddings = topic_model._extract_embeddings(documentos, method="document")

# Reatribui cada outlier ao tópico mais próximo via similaridade de vetor
new_topics = topic_model.reduce_outliers(
    documentos,
    topics,
    strategy="embeddings",
    embeddings=embeddings
)

# Atualiza o modelo e as representações de tópico com os novos rótulos
topic_model.update_topics(documentos, topics=new_topics)

# Substitui a lista de tópicos pela versão sem outliers
topics = new_topics

print(f"Outliers restantes após redução: {topics.count(-1)}")

## 7. Visualização dos tópicos

In [ ]:
topic_model.get_topic_info()

## 8. Exportação dos resultados

Adiciona a coluna `topico_modelado` ao DataFrame original e salva em CSV.

In [ ]:
arquivo_com_tópicos = "ARQUIVO COM OS TÓPICOS"  # <- altere se quiser outro nome/local

# Adiciona os tópicos ao DataFrame (garante o alinhamento correto com o índice)
comentários_do_youtube["topico_modelado"] = topics

# Salva o resultado
comentários_do_youtube.to_csv(arquivo_com_tópicos, index=False, encoding='utf-8-sig')

print(f"Arquivo salvo com sucesso em: {arquivo_com_tópicos}")
comentários_do_youtube[["COLUNA COM COMENTÁRIOS", "topico_modelado"]].head(10)